In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/nba.db")

query = """
SELECT COUNT(*) AS total_rows
FROM playoff_games
"""

df = pd.read_sql(query, conn)

display(df)

,total_rows
0,3358


In [2]:
query = """
SELECT
    MIN(GAME_DATE) AS earliest_game,
    MAX(GAME_DATE) AS latest_game
FROM playoff_games
"""

df = pd.read_sql(query, conn)

display(df)

,earliest_game,latest_game
0,1984-04-17,2026-05-15


**Comeback Rate Table**

In [3]:
conn = sqlite3.connect("../data/nba.db")

query = """
SELECT *
FROM playoff_line_scores
"""

df = pd.read_sql(query, conn)

In [4]:
score_cols = [
    "period1Score",
    "period2Score",
    "period3Score",
    "period4Score",
    "score"
]

df[score_cols] = df[score_cols].astype(int)

In [5]:
df["through_q1"] = df["period1Score"]

df["through_q2"] = (
    df["period1Score"]
    + df["period2Score"]
)

df["through_q3"] = (
    df["period1Score"]
    + df["period2Score"]
    + df["period3Score"]
)

In [11]:
q3_comebacks = []

grouped = df.groupby("gameId")

for game_id, game_df in grouped:

    if len(game_df) != 2:
        continue

    team1 = game_df.iloc[0]
    team2 = game_df.iloc[1]

    # scores entering Q4
    q3_team1 = team1["through_q3"]
    q3_team2 = team2["through_q3"]

    # final scores
    final_team1 = team1["score"]
    final_team2 = team2["score"]

    # determine who was trailing
    if q3_team1 < q3_team2:

        trailing_team = team1
        leading_team = team2

        deficit = q3_team2 - q3_team1

        trailing_won = (
            final_team1 > final_team2
        )

    else:

        trailing_team = team2
        leading_team = team1

        deficit = q3_team1 - q3_team2

        trailing_won = (
            final_team2 > final_team1
        )

    # only successful comebacks
    if trailing_won:

        q3_comebacks.append({
            "gameId": game_id,
            "Deficit Entering Q4": deficit,
            "Leading Team": leading_team["teamName"],
            "Trailing Team": trailing_team["teamName"],
            "Q3 Score":
                f'{leading_team["through_q3"]}-'
                f'{trailing_team["through_q3"]}',

            "Final Score":
                f'{leading_team["score"]}-'
                f'{trailing_team["score"]}'
        })

In [12]:
comebacks_df = pd.DataFrame(q3_comebacks)

display(comebacks_df.head())

,gameId,Deficit Entering Q4,Leading Team,Trailing Team,Q3 Score,Final Score
0,0040000001,1,Mavericks,Jazz,62-61,86-88
1,0040000002,2,Timberwolves,Spurs,65-63,82-87
2,0040000004,7,76ers,Pacers,65-58,78-79
3,0040000007,0,Kings,Suns,66-66,83-86
4,0040000030,4,Suns,Kings,63-59,82-89


In [13]:
largest_comeback = comebacks_df.sort_values(
    by="Deficit Entering Q4",
    ascending=False
)

display(largest_comeback.head(10))

,gameId,Deficit Entering Q4,Leading Team,Trailing Team,Q3 Score,Final Score
183,0041100171,21,Grizzlies,Clippers,85-64,98-99
21,0040100303,21,Nets,Celtics,74-53,90-94
227,0041400143,20,Pelicans,Warriors,89-69,119-123
313,0042000205,18,76ers,Hawks,87-69,106-109
544,0049300036,18,Rockets,Suns,100-82,117-124
161,0041000164,18,Mavericks,Trail Blazers,67-49,82-84
322,0042100153,16,Timberwolves,Grizzlies,83-67,95-104
335,0042200105,16,Bucks,Heat,102-86,126-128
518,0049100073,15,Trail Blazers,Bulls,79-64,93-97
128,0040800121,14,Magic,76ers,79-65,98-100


In [14]:
all_deficits = set()
successful_comebacks = set()

grouped = df.groupby("gameId")

for game_id, game_df in grouped:

    if len(game_df) != 2:
        continue

    team1 = game_df.iloc[0]
    team2 = game_df.iloc[1]

    q3_team1 = team1["through_q3"]
    q3_team2 = team2["through_q3"]

    final_team1 = team1["score"]
    final_team2 = team2["score"]

    deficit = abs(q3_team1 - q3_team2)

    all_deficits.add(deficit)

    trailing_team_won = (
        (q3_team1 < q3_team2 and final_team1 > final_team2)
        or
        (q3_team2 < q3_team1 and final_team2 > final_team1)
    )

    if trailing_team_won:
        successful_comebacks.add(deficit)

In [15]:
max_successful = max(successful_comebacks)

print(
    f"Largest successful Q4 playoff comeback: "
    f"{max_successful} points"
)

Largest successful Q4 playoff comeback: 21 points


In [16]:
max_observed = max(all_deficits)

never_overcome = None

for deficit in range(max_successful + 1, max_observed + 1):

    if deficit in all_deficits:

        never_overcome = deficit
        break

print(
    f"No playoff team has come back from "
    f"{never_overcome}+ entering Q4"
)

No playoff team has come back from 22+ entering Q4


In [17]:
results = []
q3_big_comebacks = []

grouped = df.groupby("gameId")

for lead_size in range(1, 22):

    q1_total = q1_leader_wins = 0
    q2_total = q2_leader_wins = 0
    q3_total = q3_leader_wins = 0

    for game_id, game_df in grouped:

        if len(game_df) != 2:
            continue

        team1 = game_df.iloc[0]
        team2 = game_df.iloc[1]

        team1_won = team1["score"] > team2["score"]

        def process_quarter(quarter_col):
            diff = abs(team1[quarter_col] - team2[quarter_col])

            if diff != lead_size:
                return None

            leader_is_team1 = team1[quarter_col] > team2[quarter_col]

            leader_won = (
                (leader_is_team1 and team1_won)
                or
                (not leader_is_team1 and not team1_won)
            )

            return leader_is_team1, leader_won, diff

        # Q1
        q1_result = process_quarter("through_q1")
        if q1_result:
            _, leader_won, _ = q1_result
            q1_total += 1
            if leader_won:
                q1_leader_wins += 1

        # Q2
        q2_result = process_quarter("through_q2")
        if q2_result:
            _, leader_won, _ = q2_result
            q2_total += 1
            if leader_won:
                q2_leader_wins += 1

        # Q3
        q3_result = process_quarter("through_q3")
        if q3_result:
            leader_is_team1, leader_won, diff = q3_result

            q3_total += 1
            if leader_won:
                q3_leader_wins += 1

            # trailing team came back from 15+
            if diff >= 15 and not leader_won:

                if leader_is_team1:
                    leading_team = team1
                    trailing_team = team2
                else:
                    leading_team = team2
                    trailing_team = team1

                q3_big_comebacks.append({
                    "gameId": game_id,
                    "Leading Team": leading_team["teamName"],
                    "Trailing Team": trailing_team["teamName"],
                    "Q3 Deficit": diff,
                    "Leading Q3 Score": leading_team["through_q3"],
                    "Trailing Q3 Score": trailing_team["through_q3"],
                    "Leading Final Score": leading_team["score"],
                    "Trailing Final Score": trailing_team["score"],
                })

    q1_losses = q1_total - q1_leader_wins
    q2_losses = q2_total - q2_leader_wins
    q3_losses = q3_total - q3_leader_wins

    results.append({
        "Lead": lead_size,

        "Through Q1":
            f"{q1_leader_wins}-{q1_losses} ({q1_leader_wins / q1_total * 100:.2f}%)"
            if q1_total > 0 else None,

        "Through Q2":
            f"{q2_leader_wins}-{q2_losses} ({q2_leader_wins / q2_total * 100:.2f}%)"
            if q2_total > 0 else None,

        "Through Q3":
            f"{q3_leader_wins}-{q3_losses} ({q3_leader_wins / q3_total * 100:.2f}%)"
            if q3_total > 0 else None,
    })

results_df = pd.DataFrame(results)
q3_big_comebacks_df = pd.DataFrame(q3_big_comebacks)

display(results_df)
display(q3_big_comebacks_df)

,Lead,Through Q1,Through Q2,Through Q3
0,1,160-165 (49.23%),120-112 (51.72%),121-105 (53.54%)
1,2,177-153 (53.64%),157-116 (57.51%),132-89 (59.73%)
2,3,204-147 (58.12%),152-93 (62.04%),132-71 (65.02%)
3,4,190-112 (62.91%),150-93 (61.73%),143-69 (67.45%)
4,5,204-99 (67.33%),136-79 (63.26%),127-60 (67.91%)
5,6,189-84 (69.23%),177-75 (70.24%),132-36 (78.57%)
6,7,140-76 (64.81%),150-55 (73.17%),176-37 (82.63%)
7,8,139-56 (71.28%),151-42 (78.24%),145-36 (80.11%)
8,9,114-43 (72.61%),142-25 (85.03%),135-24 (84.91%)
9,10,106-31 (77.37%),122-34 (78.21%),127-12 (91.37%)


,gameId,Leading Team,Trailing Team,Q3 Deficit,Leading Q3 Score,Trailing Q3 Score,Leading Final Score,Trailing Final Score
0,0049100073,Trail Blazers,Bulls,15,79,64,93,97
1,0042100153,Timberwolves,Grizzlies,16,83,67,95,104
2,0042200105,Bucks,Heat,16,102,86,126,128
3,0041000164,Mavericks,Trail Blazers,18,67,49,82,84
4,0042000205,76ers,Hawks,18,87,69,106,109
5,0049300036,Rockets,Suns,18,100,82,117,124
6,0041400143,Pelicans,Warriors,20,89,69,119,123
7,0040100303,Nets,Celtics,21,74,53,90,94
8,0041100171,Grizzlies,Clippers,21,85,64,98,99


In [18]:
comeback_game_ids = q3_big_comebacks_df["gameId"].unique().tolist()

if comeback_game_ids:
    placeholders = ",".join(["?"] * len(comeback_game_ids))

    metadata_query = f"""
    SELECT
        GAME_ID,
        GAME_DATE,
        MATCHUP,
        TEAM_NAME,
        TEAM_ABBREVIATION,
        WL,
        PTS,
        PLUS_MINUS
    FROM playoff_games
    WHERE GAME_ID IN ({placeholders})
    ORDER BY GAME_DATE, GAME_ID
    """

    comeback_metadata_df = pd.read_sql(
        metadata_query,
        conn,
        params=comeback_game_ids
    )

    display(comeback_metadata_df)
else:
    print("No Q3 15+ comeback games found.")

,game_id,game_date,matchup,team_name,team_abbreviation,wl,pts,plus_minus
0,0049100073,1992-06-14,POR @ CHI,Portland Trail Blazers,POR,L,93,NaN
1,0049300036,1994-05-11,HOU vs. PHX,Houston Rockets,HOU,L,117,NaN
2,0040100303,2002-05-25,NJN @ BOS,New Jersey Nets,NJN,L,90,-4.0
3,0041000164,2011-04-23,POR vs. DAL,Portland Trail Blazers,POR,W,84,2.0
4,0041100171,2012-04-29,MEM vs. LAC,Memphis Grizzlies,MEM,L,98,-1.0
5,0041400143,2015-04-23,GSW @ NOP,Golden State Warriors,GSW,W,123,4.0
6,0042000205,2021-06-16,ATL @ PHI,Atlanta Hawks,ATL,W,109,3.0
7,0042100153,2022-04-21,MIN vs. MEM,Minnesota Timberwolves,MIN,L,95,-9.0
8,0042200105,2023-04-26,MIA @ MIL,Miami Heat,MIA,W,128,2.0


In [20]:
combined_df = q3_big_comebacks_df.merge(
    comeback_metadata_df,
    left_on="gameId",
    right_on="game_id",
    how="left"
)

display(combined_df)

,gameId,Leading Team,Trailing Team,Q3 Deficit,Leading Q3 Score,Trailing Q3 Score,Leading Final Score,Trailing Final Score,game_id,game_date,matchup,team_name,team_abbreviation,wl,pts,plus_minus
0,0049100073,Trail Blazers,Bulls,15,79,64,93,97,0049100073,1992-06-14,POR @ CHI,Portland Trail Blazers,POR,L,93,NaN
1,0042100153,Timberwolves,Grizzlies,16,83,67,95,104,0042100153,2022-04-21,MIN vs. MEM,Minnesota Timberwolves,MIN,L,95,-9.0
2,0042200105,Bucks,Heat,16,102,86,126,128,0042200105,2023-04-26,MIA @ MIL,Miami Heat,MIA,W,128,2.0
3,0041000164,Mavericks,Trail Blazers,18,67,49,82,84,0041000164,2011-04-23,POR vs. DAL,Portland Trail Blazers,POR,W,84,2.0
4,0042000205,76ers,Hawks,18,87,69,106,109,0042000205,2021-06-16,ATL @ PHI,Atlanta Hawks,ATL,W,109,3.0
5,0049300036,Rockets,Suns,18,100,82,117,124,0049300036,1994-05-11,HOU vs. PHX,Houston Rockets,HOU,L,117,NaN
6,0041400143,Pelicans,Warriors,20,89,69,119,123,0041400143,2015-04-23,GSW @ NOP,Golden State Warriors,GSW,W,123,4.0
7,0040100303,Nets,Celtics,21,74,53,90,94,0040100303,2002-05-25,NJN @ BOS,New Jersey Nets,NJN,L,90,-4.0
8,0041100171,Grizzlies,Clippers,21,85,64,98,99,0041100171,2012-04-29,MEM vs. LAC,Memphis Grizzlies,MEM,L,98,-1.0
